In [ ]:
!pip install -q requests beautifulsoup4 boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 6.7 MB/s eta 0:00:00


In [ ]:
import boto3, getpass

In [ ]:
session = boto3.Session(
    aws_access_key_id=getpass.getpass(''),
    aws_secret_access_key=getpass.getpass(''),
    aws_session_token=getpass.getpass(''),
    region_name='ap-southeast-2'
)

··········
··········
··········


In [ ]:
print(session.client("sts").get_caller_identity())

{'UserId': 'AROA42GGNDVHU7ITXZWIX:ramonworkshop', 'Account': '880884391247', 'Arn': 'arn:aws:sts::880884391247:assumed-role/AWSReservedSSO_AdministratorAccess_4c10792d593bbc38/ramonworkshop', 'ResponseMetadata': {'RequestId': '62625aea-c62d-4f10-ae69-8dd4d61756f3', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': '62625aea-c62d-4f10-ae69-8dd4d61756f3', 'x-amz-sts-extended-request-id': 'MTphcC1zb3V0aGVhc3QtMjpTOjE3OTEwOTczMzIxOTU6UjpCcVIxRTl3eQ==', 'content-type': 'text/xml', 'content-length': '484', 'date': 'Sun, 04 Oct 2026 07:02:12 GMT'}, 'RetryAttempts': 0}}


In [ ]:
from bs4 import BeautifulSoup as bs
import requests
import json
from datetime import datetime, timezone

In [ ]:

BUCKET = "ramon-de-workshop-2026"
url = "https://books.toscrape.com/catalogue/category/books_1/page-{}.html"

def scrape(pages=2):
  books = []

  try:
    for n in range(1, pages + 1):
      req = requests.get(url.format(n), timeout = 10)
      req.raise_for_status()
      soup = bs(req.text, "html.parser")
      for item in soup.select("article.product_pod"):
        books.append({
            "title": item.select_one("h3 a")["title"],
            "price": item.select_one("div.product_price p.price_color").text,
            "rating": item.select_one("p")["class"]
        })
    return books
  except requests.exceptions.HTTPError:
    print("Error occured.")
  else:
    print("\nRequest Success!")

def convert(books):
  json_books = []
  for book in books:
    json_books.append(json.dumps(book))
  json_books = "\n".join(json_books)
  return json_books

def land(json_books):
  now = datetime.now(timezone.utc)
  filepath = f"raw/books/{now.year}/{now.month}-{now.day}/{now.hour}:{now.minute}:{now.second}.json"
  s3 = session.client("s3")
  s3.put_object(
      Bucket=BUCKET,
      Key=filepath,
      Body=json_books
  )
books = scrape()

land(convert(books))


In [ ]:
import pandas as pd

In [ ]:
def clean(books):
  df = pd.DataFrame(books)

  df["price"] = df["price"].str.replace("Â£", "")

  df["rating"] = df["rating"].str[1]

  dictionary = {
      "One": "1",
      "Two": "2",
      "Three": "3",
      "Four": "4",
      "Five": "5"
  }

  df["rating"] = df["rating"].map(dictionary)

  df["price"] = df["price"].astype(float)
  df["rating"] = df["rating"].astype(int)

  return df

clean(books)


In [ ]:
def land_clean(BUCKET, df):
  bucket = BUCKET
  now = datetime.now(timezone.utc)
  body = df.to_parquet(index=False)
  filepath = f"CLEAN/books/{now.year}/{now.month}-{now.day}/{now.hour}:{now.minute}:{now.second}.parquet"
  s3 = session.client("s3")
  s3.put_object(
      Bucket = bucket,
      Key = filepath,
      Body = body
  )
land_clean(BUCKET, clean(books))


SQL Queries:

In [ ]:
CREATE DATABASE workshop_db;

CREATE EXTERNAL TABLE IF NOT EXISTS books (
  title string,
  price double,
  rating bigint
)
STORED AS PARQUET
LOCATION 's3://workshop-angeles-2026/CLEAN/';

SELECT * FROM workshopdb.books LIMIT 10;